# Generalized Linear Models: Negative Binomial Regression

This notebook demonstrates Negative Binomial Regression, a GLM for count data with overdispersion, where variance exceeds the mean.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)

In [ ]:
# Generate count data with overdispersion
n_samples = 200
X = np.random.randn(n_samples, 3)
true_coef = np.array([1.5, -2.0, 0.5])
eta = X @ true_coef
mu = np.exp(eta)

# Negative binomial with overdispersion parameter
alpha = 2.0  # Overdispersion
y = np.random.negative_binomial(n=1/alpha, p=1/(1 + mu*alpha))

df = pd.DataFrame(X, columns=['X1', 'X2', 'X3'])
df['counts'] = y

print(f"Data shape: {df.shape}")
print(f"Count statistics:")
print(df['counts'].describe())
print(f"\nMean: {y.mean():.2f}, Variance: {y.var():.2f}")
print(f"Variance/Mean ratio (overdispersion): {y.var()/y.mean():.2f}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Poisson regressor
poisson = PoissonRegressor(alpha=0.0, max_iter=1000)
poisson.fit(X_train, y_train)
y_pred_poisson = poisson.predict(X_test)

# Negative binomial via Tweedie (power=2)
negbin = TweedieRegressor(power=2, alpha=0.0, max_iter=1000)
negbin.fit(X_train, y_train)
y_pred_negbin = negbin.predict(X_test)

# Metrics
mse_poisson = mean_squared_error(y_test, y_pred_poisson)
mse_negbin = mean_squared_error(y_test, y_pred_negbin)

mae_poisson = mean_absolute_error(y_test, y_pred_poisson)
mae_negbin = mean_absolute_error(y_test, y_pred_negbin)

print("\n=== POISSON vs. NEGATIVE BINOMIAL ===")
print(f"Poisson MSE: {mse_poisson:.4f}, NegBin MSE: {mse_negbin:.4f}")
print(f"Poisson MAE: {mae_poisson:.4f}, NegBin MAE: {mae_negbin:.4f}")
print(f"\nNegative Binomial advantage (MSE): {(mse_poisson - mse_negbin)/mse_poisson * 100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred_poisson, alpha=0.6, label='Poisson')
axes[0].scatter(y_test, y_pred_negbin, alpha=0.6, label='Negative Binomial')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('True Counts')
axes[0].set_ylabel('Predicted Counts')
axes[0].set_title('Poisson vs. Negative Binomial Predictions')
axes[0].legend()
axes[0].grid(alpha=0.3)

residuals_poisson = y_test - y_pred_poisson
residuals_negbin = y_test - y_pred_negbin

axes[1].scatter(y_pred_poisson, residuals_poisson, alpha=0.6, label='Poisson')
axes[1].scatter(y_pred_negbin, residuals_negbin, alpha=0.6, label='Negative Binomial')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Comparison')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("""
Negative Binomial Regression:
- Handles overdispersed count data (variance > mean)
- More flexible than Poisson
- Better for real-world count data with extra variation
- Models both mean and dispersion
- Useful when Poisson assumptions violated
""")